# Trigger sweep on Barrett reduction

This notebook finds which operation of HQC's Barrett reduction leaks. `barrett_sweep.c` writes the reduction with one operation per line. Each test builds the board code with the trigger set low one operation earlier than the test before, so the recorded window gets smaller, then runs the fixed-vs-random Welch t-test. `TRIGGER_STAGE` in `simpleserial-barrett-sweep/barrett_sweep.c` picks the point:

| Test | Trigger low after |
| ---- | ----------------- |
| 1 | `r_out = barrett_reduce(x_in)`, the normal position |
| 2 | `r = r - sel` |
| 3 | `sel = mask & N` |
| 4 | `mask = -reduce_flag` |
| 5 | `reduce_flag = sign ^ 1` |
| 6 | `sign = t >> 31` |
| 7 | `t = r - N` |
| 8 | `r = x - qn` |
| 9 | `qn = q * N` |
| 10 | `q = (x * mu) >> 32` |
| 11 | reading `x` |

## How to read the tests

- Each test first prints the board instructions of `handle_barrett`. The trigger window, from cycle 0 to the vertical line in the plots, runs from the `str` in `pio_set_pin_high` that sets the pin high to the `str` in `pio_set_pin_low` that sets it low. It has three parts:
  1. The end of `trigger_high`, after the pin rises: its return and the call to `pio_configure_pin`. This takes about the first 100 cycles and is the same in every test.
  2. The Barrett part, in `handle_barrett`, from after the `blx` to `trigger_high` up to the `blx` to `trigger_low`. Only this part changes between tests. The line under each listing names its instructions. The `ldr` just before the second `blx` only loads the address of `trigger_low`.
  3. The start of `trigger_low`, before the pin falls: `push {r4, lr}`, `ldr`, `movs`, `blx` and seven instructions in `pio_set_pin_low` up to the `str`. This takes roughly the last 15 cycles and is the same in every test.
- The plots count clock cycles from trigger high, so trigger high is cycle 0. The vertical line marks trigger low.
- Code before the `trigger_low` call shows before the line. The recording goes on after trigger low, so the code after the call shows after the line, a little later than in test 1 because the call takes time.
- Go through the tests in order. Each test takes one operation out of the window. If a leaking cycle lies before the line in one test but not in the next, it comes from the operation that the next test took out.
- Test 11 keeps only the read of `x_in`. x is the value that differs between the two groups, so a leak before its line comes from reading x.
- A leak that sits the same number of cycles before the line in every test, and moves with it, comes from the `trigger_low` call, not from Barrett.
- Without a cut, the one-operation lines compile to the same instructions as HQC's code (test 1). A cut between two operations that the compiler normally merges changes the instructions: `mvns` and `lsrs` in test 6, and one `mls` in test 9. Their parts say how.
- `trigger_low` saves register `r4` on the stack, roughly 15 cycles before trigger low. After the pin goes low, it calls `pio_configure_pin`, which saves and restores `r4` again. In tests 2 to 11, `r4` holds the value just computed (r, sel, mask, reduce_flag, sign, t, r, qn, q or x), so both saves can leak on their own, just before and just after the line. In test 1, `r4` holds the buffer address.

## Why trigger high stays in the same place

- The Husky starts recording when the trigger goes high. Its sampling clock is locked to the board clock, so the same instruction lands on the same sample in every run. Trigger high is the time reference of every plot.
- If trigger high moved, a different instruction would sit at cycle 0, and every leak would move to a new cycle number. The tests could no longer be compared cycle by cycle.
- `trigger_high()` is code too: a function call that writes the pin register. Putting it inside `barrett_reduce` would split the reduction the same way `trigger_low()` does here. The compiler keeps values in other registers across the call and can reorder the instructions, so the code under test changes, and a leak could come from the new code instead of from Barrett.
- With trigger high fixed, the code from trigger high to the chosen line is nearly the same in every test, so a leak keeps about the same cycle number. Only trigger low moves. The instruction listings show the small register changes the compiler makes in each test.

In [1]:
import os
import subprocess

import numpy as np
import chipwhisperer as cw
import holoviews as hv
from scipy.stats import ttest_ind
from tqdm.auto import tqdm

N = 17669           # the modulus, must be the same as PARAM_N in barrett_sweep.c
FIXED_X = 1212218   # the input used for every fixed-group run
N_WARMUP = 1000     # runs done once before test 1 to warm up the board, their traces are not kept
N_TRACES = 5000     # runs per test, fixed and random together
SAMPLES = 1000      # power samples recorded in each run
SEED = 1            # the same seed gives the same inputs every time
THRESHOLD = 4.5     # the |t| limit, above it in both halves means a leak

FIRMWARE = "simpleserial-barrett-sweep"
HEX = FIRMWARE + "/simpleserial-barrett-sweep-CWHUSKY.hex"
ELF = FIRMWARE + "/simpleserial-barrett-sweep-CWHUSKY.elf"
FIGURES = "figures/trigger-sweep"  # folder for the PNG copies of the plots

# Bokeh shows the interactive plots, matplotlib draws the PNG copies
hv.extension("bokeh", "matplotlib", logo=False)

## Setup

Connects to the Husky and the target board, and finds the clock cycle of each sample, counted from trigger high.

In [2]:
scope = cw.scope()
target = cw.target(scope)
scope.default_setup()
scope.adc.samples = SAMPLES

SAMPLES_PER_CYCLE = round(scope.clock.adc_freq / scope.clock.clkgen_freq)
cycles = (np.arange(SAMPLES) - scope.adc.presamples) / SAMPLES_PER_CYCLE

scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.glitch.phase_

## Inputs

The groups alternate: fixed, random, fixed, random, and so on. Every test uses the same inputs.

In [3]:
# 0 = fixed group (input FIXED_X), 1 = random group (a random 24-bit number)
groups = np.arange(N_TRACES) % 2
rng = np.random.default_rng(SEED)
inputs = np.where(groups == 0, FIXED_X, rng.integers(0, 2**24, N_TRACES))
warmup_inputs = rng.integers(0, 2**24, N_WARMUP)

## Functions

- `build_and_load(test)` builds the board code for one test and loads it onto the board.
- `warm_up()` runs the reduction `N_WARMUP` times so the board reaches a steady temperature. It runs once, in test 1.
- `record()` records one trace for every input.
- `welch_all` runs the Welch t-test on all traces. It is used in test 1 only.
- `welch_halves` runs the Welch t-test on each half of the traces. A sample leaks only if both halves cross `THRESHOLD` with the same sign.

In [4]:
# Runs make in the firmware folder. Stops with make's error output if it fails.
def make(*args):
    result = subprocess.run(["make", "-C", FIRMWARE, *args], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"make {' '.join(args)} failed:\n{result.stderr}")


# Builds the board code with trigger low at point `test` and loads it onto the board.
def build_and_load(test):
    make("clean")
    make(f"TRIGGER_STAGE={test}")
    cw.program_target(scope, cw.programmers.SAM4SProgrammer, HEX)

In [5]:
# Sends x to the board, runs one reduction, and returns the recorded power.
# Stops with an error if nothing is recorded or the board gives a wrong answer.
def capture(x):
    scope.arm()
    target.simpleserial_write("b", bytearray(x.to_bytes(3, "little")))
    if scope.capture():
        raise RuntimeError(f"x={x}: scope timed out, no trace recorded")
    reply = target.simpleserial_read("r", 4)
    if reply is None or int.from_bytes(reply, "little") != x % N:
        raise RuntimeError(f"x={x}: board sent {reply}, expected {x % N}")
    return scope.get_last_trace()


# Runs the reduction on the warm-up inputs and throws the traces away.
def warm_up():
    for x in tqdm(warmup_inputs, desc="warm-up"):
        capture(int(x))


# Records one trace per input. Returns the traces and the cycle of trigger low.
def record():
    # Record one trace to read the trigger length; the recording must reach trigger low
    capture(FIXED_X)
    if scope.adc.presamples + scope.adc.trig_count > SAMPLES:
        raise ValueError(f"set SAMPLES to at least {scope.adc.presamples + scope.adc.trig_count}")
    trigger_low = scope.adc.trig_count / SAMPLES_PER_CYCLE
    print(f"trigger low at cycle {trigger_low:.0f}")

    traces = np.array([capture(int(x)) for x in tqdm(inputs, desc="capture")])
    return traces, trigger_low

In [6]:
# Returns the Welch t-value at each sample between the traces in a and in b.
def welch_t(a, b):
    return ttest_ind(a, b, axis=0, equal_var=False).statistic


# Plots each t-value curve in curves ({label: t}) against the cycles, with lines
# at +THRESHOLD and -THRESHOLD and a vertical line at trigger low.
def plot_t(curves, trigger_low):
    xlabel = "cycles from trigger high"
    bound = np.full(SAMPLES, THRESHOLD)
    figure = cw.plot((cycles, bound), xlabel, "t") * cw.plot((cycles, -bound), xlabel, "t")
    for label, t in curves.items():
        figure *= cw.plot((cycles, t), xlabel, "t", label=label)
    return figure * hv.VLine(trigger_low)


# Saves a PNG copy of a plot as FIGURES/<name>.png.
# Returns the plot unchanged, so the cell still shows the interactive version.
def save_png(figure, name):
    os.makedirs(FIGURES, exist_ok=True)
    wide = figure.opts(hv.opts.Curve(linewidth=0.8, aspect=2, fig_inches=10, backend="matplotlib"),
                       hv.opts.Overlay(aspect=2, fig_inches=10, backend="matplotlib"), clone=True)
    hv.save(wide, os.path.join(FIGURES, name + ".png"), backend="matplotlib", dpi=100)
    return figure

In [7]:
# Runs the Welch t-test on all traces, prints the largest |t|,
# plots it, and saves a PNG copy as FIGURES/<name>.png.
def welch_all(traces, trigger_low, name):
    t = welch_t(traces[groups == 0], traces[groups == 1])
    print(f"max |t|: {np.max(np.abs(t)):.1f}")
    return save_png(plot_t({"all traces": t}, trigger_low), name)


# Runs the Welch t-test on each half of the traces, prints the leaking cycles,
# plots both halves, and saves a PNG copy as FIGURES/<name>.png.
def welch_halves(traces, trigger_low, name):
    fixed = traces[groups == 0]
    random = traces[groups == 1]
    half = len(fixed) // 2
    t1 = welch_t(fixed[:half], random[:half])
    t2 = welch_t(fixed[half:], random[half:])

    # A sample leaks when |t| is above the threshold in both halves, with the same sign
    leak = (np.abs(t1) > THRESHOLD) & (np.abs(t2) > THRESHOLD) & (np.sign(t1) == np.sign(t2))
    leak_cycles = np.unique(np.floor(cycles[leak]).astype(int))
    print("leaking cycles before trigger low:", leak_cycles[leak_cycles < trigger_low].tolist())
    print("leaking cycles after trigger low:", leak_cycles[leak_cycles >= trigger_low].tolist())
    return save_png(plot_t({"half 1": t1, "half 2": t2}, trigger_low), name)

## Test 1: trigger low after `r_out` gets its value (normal)

In `handle_barrett`, right after `r_out = barrett_reduce(x_in);`, the same place as in `simpleserial-barrett.c`. On the board, `trigger_low` is called after the `str` that writes `r_out`.

In [8]:
build_and_load(1)

In [9]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett-sweep/simpleserial-barrett-sweep-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b538      	push	{r3, r4, r5, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4d16      	ldr	r5, [pc, #88]	@ (4001b8 <handle_barrett+0x60>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	602b      	str	r3, [r5, #0]
  40016c:	4b13      	ldr	r3, [pc, #76]	@ (4001bc <handle_barrett+0x64>)
  40016e:	4604      	mov	r4, r0
  400170:	4798      	blx	r3
  400172:	682a      	ldr	r2, [r5, #0]
  400174:	4b12      	ldr	r3, [pc, #72]	@ (4001c0 <handle_barrett+0x68>)
  400176:	4d13      	ldr	r5, [pc, #76]	@ (4001c4 <handle_barrett+0x6c>)
  400178:	fba3 1302 	umull	r1, r3, r3, r2
  40017c:	f244 5105 	movw	r1, #17669	@ 0x4505
  400180:	fb01 2213 	mls	r2, r1, r3, r2
  400184:	f

Barrett part of the trigger window (`handle_barrett`, between the `blx` to `trigger_high` and the `blx` to `trigger_low`): `ldr` (read x), `ldr` (load mu), `ldr` (address of `r_out`), `umull`, `movw` (load N), `mls`, `sub.w`, `subs`, `mvns`, `lsrs`, `negs`, `ands`, `subs`, `ldr` (address of `trigger_low`), `str` (write `r_out`).

In [10]:
warm_up()

warm-up:   0%|          | 0/1000 [00:00<?, ?it/s]

In [11]:
traces, trigger_low = record()

trigger low at cycle 140


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [12]:
welch_all(traces, trigger_low, "test1_welch")

max |t|: 74.5


:Overlay
   .Curve.I          :Curve   [cycles from trigger high]   (t)
   .Curve.II         :Curve   [cycles from trigger high]   (t)
   .Curve.All_traces :Curve   [cycles from trigger high]   (t)
   .VLine.I          :VLine   [x,y]

![Welch t-test on all traces, test 1](figures/trigger-sweep/test1_welch.png)

In [13]:
welch_halves(traces, trigger_low, "test1_halves")

leaking cycles before trigger low: [102, 105, 106, 107, 110, 111, 112, 113, 114, 115, 116, 118, 119, 123, 124]
leaking cycles after trigger low: [244, 245, 246, 247, 248, 249]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 1](figures/trigger-sweep/test1_halves.png)

## Test 2: trigger low after `r = r - sel`

In `barrett_reduce`, after `r = r - sel;`. On the board, after the last `subs` and before the `str` that writes `r_out`. The store of `r_out` leaves the window.

In [14]:
build_and_load(2)

In [15]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett-sweep/simpleserial-barrett-sweep-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b538      	push	{r3, r4, r5, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4c16      	ldr	r4, [pc, #88]	@ (4001b8 <handle_barrett+0x60>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	6023      	str	r3, [r4, #0]
  40016c:	4b13      	ldr	r3, [pc, #76]	@ (4001bc <handle_barrett+0x64>)
  40016e:	4605      	mov	r5, r0
  400170:	4798      	blx	r3
  400172:	6824      	ldr	r4, [r4, #0]
  400174:	4b12      	ldr	r3, [pc, #72]	@ (4001c0 <handle_barrett+0x68>)
  400176:	fba3 2304 	umull	r2, r3, r3, r4
  40017a:	f244 5205 	movw	r2, #17669	@ 0x4505
  40017e:	fb02 4413 	mls	r4, r2, r3, r4
  400182:	f5a4 438a 	sub.w	r3, r4, #17664	@ 0x4500
  400186:	3b05      	subs	r3, #5

Barrett part of the trigger window (`handle_barrett`, between the `blx` to `trigger_high` and the `blx` to `trigger_low`): `ldr` (read x), `ldr` (load mu), `umull`, `movw` (load N), `mls`, `sub.w`, `subs`, `mvns`, `lsrs`, `negs`, `ands`, `subs`, `ldr` (address of `trigger_low`).

In [16]:
traces, trigger_low = record()

trigger low at cycle 139


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [17]:
welch_halves(traces, trigger_low, "test2_halves")

leaking cycles before trigger low: [102, 104, 105, 107, 109, 110, 111, 112, 113, 114, 121, 122]
leaking cycles after trigger low: [142, 143, 248, 249]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 2](figures/trigger-sweep/test2_halves.png)

## Test 3: trigger low after `sel = mask & N`

In `barrett_reduce`, after `uint32_t sel = mask & PARAM_N;`. On the board, after `ands` and before the last `subs`. The final subtraction leaves the window.

In [18]:
build_and_load(3)

In [19]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett-sweep/simpleserial-barrett-sweep-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b570      	push	{r4, r5, r6, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4c16      	ldr	r4, [pc, #88]	@ (4001b8 <handle_barrett+0x60>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	6023      	str	r3, [r4, #0]
  40016c:	4b13      	ldr	r3, [pc, #76]	@ (4001bc <handle_barrett+0x64>)
  40016e:	4605      	mov	r5, r0
  400170:	4798      	blx	r3
  400172:	6826      	ldr	r6, [r4, #0]
  400174:	4b12      	ldr	r3, [pc, #72]	@ (4001c0 <handle_barrett+0x68>)
  400176:	fba3 2306 	umull	r2, r3, r3, r6
  40017a:	f244 5205 	movw	r2, #17669	@ 0x4505
  40017e:	fb02 6613 	mls	r6, r2, r3, r6
  400182:	f5a6 438a 	sub.w	r3, r6, #17664	@ 0x4500
  400186:	3b05      	subs	r3, #5

Barrett part of the trigger window (`handle_barrett`, between the `blx` to `trigger_high` and the `blx` to `trigger_low`): `ldr` (read x), `ldr` (load mu), `umull`, `movw` (load N), `mls`, `sub.w`, `subs`, `mvns`, `lsrs`, `negs`, `ands`, `ldr` (address of `trigger_low`).

In [20]:
traces, trigger_low = record()

trigger low at cycle 138


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [21]:
welch_halves(traces, trigger_low, "test3_halves")

leaking cycles before trigger low: [102, 104, 105, 107, 108, 109, 110, 111, 112, 113, 120, 121, 122]
leaking cycles after trigger low: [242, 243, 248, 249]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 3](figures/trigger-sweep/test3_halves.png)

## Test 4: trigger low after `mask = -reduce_flag`

In `barrett_reduce`, after `uint32_t mask = -reduce_flag;`. On the board, after `negs` and before `ands`. `sel` leaves the window.

In [22]:
build_and_load(4)

In [23]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett-sweep/simpleserial-barrett-sweep-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b5f8      	push	{r3, r4, r5, r6, r7, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4c16      	ldr	r4, [pc, #88]	@ (4001b8 <handle_barrett+0x60>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	6023      	str	r3, [r4, #0]
  40016c:	4b13      	ldr	r3, [pc, #76]	@ (4001bc <handle_barrett+0x64>)
  40016e:	4605      	mov	r5, r0
  400170:	4798      	blx	r3
  400172:	6826      	ldr	r6, [r4, #0]
  400174:	4b12      	ldr	r3, [pc, #72]	@ (4001c0 <handle_barrett+0x68>)
  400176:	f244 5705 	movw	r7, #17669	@ 0x4505
  40017a:	fba3 2306 	umull	r2, r3, r3, r6
  40017e:	fb07 6613 	mls	r6, r7, r3, r6
  400182:	f5a6 448a 	sub.w	r4, r6, #17664	@ 0x4500
  400186:	3c05      	sub

Barrett part of the trigger window (`handle_barrett`, between the `blx` to `trigger_high` and the `blx` to `trigger_low`): `ldr` (read x), `ldr` (load mu), `movw` (load N), `umull`, `mls`, `sub.w`, `subs`, `mvns`, `lsrs`, `negs`, `ldr` (address of `trigger_low`).

In [24]:
traces, trigger_low = record()

trigger low at cycle 137


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [25]:
welch_halves(traces, trigger_low, "test4_halves")

leaking cycles before trigger low: [102, 105, 106, 107, 110, 112, 113, 119, 120, 121]
leaking cycles after trigger low: [242, 243, 248, 249]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 4](figures/trigger-sweep/test4_halves.png)

## Test 5: trigger low after `reduce_flag = sign ^ 1`

In `barrett_reduce`, after `uint32_t reduce_flag = sign ^ 1;`. On the board, after `lsrs` and before `negs`. The compiler computes `sign` and `reduce_flag` together, with `mvns` (flip all bits) and then `lsrs` (keep the top bit). `mask` leaves the window.

In [26]:
build_and_load(5)

In [27]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett-sweep/simpleserial-barrett-sweep-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b5f8      	push	{r3, r4, r5, r6, r7, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4c16      	ldr	r4, [pc, #88]	@ (4001b8 <handle_barrett+0x60>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	6023      	str	r3, [r4, #0]
  40016c:	4b13      	ldr	r3, [pc, #76]	@ (4001bc <handle_barrett+0x64>)
  40016e:	4605      	mov	r5, r0
  400170:	4798      	blx	r3
  400172:	6826      	ldr	r6, [r4, #0]
  400174:	4b12      	ldr	r3, [pc, #72]	@ (4001c0 <handle_barrett+0x68>)
  400176:	f244 5705 	movw	r7, #17669	@ 0x4505
  40017a:	fba3 2306 	umull	r2, r3, r3, r6
  40017e:	fb07 6613 	mls	r6, r7, r3, r6
  400182:	f5a6 448a 	sub.w	r4, r6, #17664	@ 0x4500
  400186:	3c05      	sub

Barrett part of the trigger window (`handle_barrett`, between the `blx` to `trigger_high` and the `blx` to `trigger_low`): `ldr` (read x), `ldr` (load mu), `movw` (load N), `umull`, `mls`, `sub.w`, `subs`, `mvns`, `lsrs`, `ldr` (address of `trigger_low`).

In [28]:
traces, trigger_low = record()

trigger low at cycle 136


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [29]:
welch_halves(traces, trigger_low, "test5_halves")

leaking cycles before trigger low: [98, 102, 105, 106, 107, 109, 110, 112, 114, 118, 119, 120, 121]
leaking cycles after trigger low: [242, 243, 246, 248, 249]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 5](figures/trigger-sweep/test5_halves.png)

## Test 6: trigger low after `sign = t >> 31`

In `barrett_reduce`, after `uint32_t sign = t >> 31;`. On the board, after `lsrs` and before `eor.w #1`. The cut changes the instructions: instead of `mvns` before `lsrs`, the compiler flips the bit with `eor.w r4, r4, #1` after the call. `reduce_flag` leaves the window.

In [30]:
build_and_load(6)

In [31]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett-sweep/simpleserial-barrett-sweep-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b5f8      	push	{r3, r4, r5, r6, r7, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4c17      	ldr	r4, [pc, #92]	@ (4001bc <handle_barrett+0x64>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	6023      	str	r3, [r4, #0]
  40016c:	4b14      	ldr	r3, [pc, #80]	@ (4001c0 <handle_barrett+0x68>)
  40016e:	4605      	mov	r5, r0
  400170:	4798      	blx	r3
  400172:	6826      	ldr	r6, [r4, #0]
  400174:	4b13      	ldr	r3, [pc, #76]	@ (4001c4 <handle_barrett+0x6c>)
  400176:	f244 5705 	movw	r7, #17669	@ 0x4505
  40017a:	fba3 2306 	umull	r2, r3, r3, r6
  40017e:	fb07 6613 	mls	r6, r7, r3, r6
  400182:	f5a6 448a 	sub.w	r4, r6, #17664	@ 0x4500
  400186:	3c05      	sub

Barrett part of the trigger window (`handle_barrett`, between the `blx` to `trigger_high` and the `blx` to `trigger_low`): `ldr` (read x), `ldr` (load mu), `movw` (load N), `umull`, `mls`, `sub.w`, `subs`, `lsrs`, `ldr` (address of `trigger_low`).

In [32]:
traces, trigger_low = record()

trigger low at cycle 135


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [33]:
welch_halves(traces, trigger_low, "test6_halves")

leaking cycles before trigger low: [6, 102, 105, 106, 107, 109, 110, 111, 117, 118, 119]
leaking cycles after trigger low: [243, 244, 249]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 6](figures/trigger-sweep/test6_halves.png)

## Test 7: trigger low after `t = r - N`

In `barrett_reduce`, after `uint32_t t = r - PARAM_N;`. On the board, after `sub.w #17664` and `subs #5`, which subtract N in two steps because 17669 does not fit in one instruction. `sign` leaves the window.

In [34]:
build_and_load(7)

In [35]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett-sweep/simpleserial-barrett-sweep-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b5f8      	push	{r3, r4, r5, r6, r7, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4c16      	ldr	r4, [pc, #88]	@ (4001b8 <handle_barrett+0x60>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	6023      	str	r3, [r4, #0]
  40016c:	4b13      	ldr	r3, [pc, #76]	@ (4001bc <handle_barrett+0x64>)
  40016e:	4605      	mov	r5, r0
  400170:	4798      	blx	r3
  400172:	6826      	ldr	r6, [r4, #0]
  400174:	4b12      	ldr	r3, [pc, #72]	@ (4001c0 <handle_barrett+0x68>)
  400176:	f244 5705 	movw	r7, #17669	@ 0x4505
  40017a:	fba3 2306 	umull	r2, r3, r3, r6
  40017e:	fb07 6613 	mls	r6, r7, r3, r6
  400182:	f5a6 448a 	sub.w	r4, r6, #17664	@ 0x4500
  400186:	3c05      	sub

Barrett part of the trigger window (`handle_barrett`, between the `blx` to `trigger_high` and the `blx` to `trigger_low`): `ldr` (read x), `ldr` (load mu), `movw` (load N), `umull`, `mls`, `sub.w`, `subs`, `ldr` (address of `trigger_low`).

In [36]:
traces, trigger_low = record()

trigger low at cycle 134


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [37]:
welch_halves(traces, trigger_low, "test7_halves")

leaking cycles before trigger low: [102, 105, 106, 107, 109, 110, 117, 118]
leaking cycles after trigger low: [151, 152, 241, 242, 243, 248, 249]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 7](figures/trigger-sweep/test7_halves.png)

## Test 8: trigger low after `r = x - qn`

In `barrett_reduce`, after `uint32_t r = x - qn;`. On the board, after `mls` and before `sub.w`. `t` leaves the window.

In [38]:
build_and_load(8)

In [39]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett-sweep/simpleserial-barrett-sweep-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b570      	push	{r4, r5, r6, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4c17      	ldr	r4, [pc, #92]	@ (4001bc <handle_barrett+0x64>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	6023      	str	r3, [r4, #0]
  40016c:	4b14      	ldr	r3, [pc, #80]	@ (4001c0 <handle_barrett+0x68>)
  40016e:	4605      	mov	r5, r0
  400170:	4798      	blx	r3
  400172:	6823      	ldr	r3, [r4, #0]
  400174:	4a13      	ldr	r2, [pc, #76]	@ (4001c4 <handle_barrett+0x6c>)
  400176:	f244 5605 	movw	r6, #17669	@ 0x4505
  40017a:	fba2 1203 	umull	r1, r2, r2, r3
  40017e:	fb06 3412 	mls	r4, r6, r2, r3
  400182:	4b11      	ldr	r3, [pc, #68]	@ (4001c8 <handle_barrett+0x70>)
  400184:	4

Barrett part of the trigger window (`handle_barrett`, between the `blx` to `trigger_high` and the `blx` to `trigger_low`): `ldr` (read x), `ldr` (load mu), `movw` (load N), `umull`, `mls`, `ldr` (address of `trigger_low`).

In [40]:
traces, trigger_low = record()

trigger low at cycle 132


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [41]:
welch_halves(traces, trigger_low, "test8_halves")

leaking cycles before trigger low: [6, 10, 11, 20, 102, 105, 106, 107, 108, 114, 115, 116, 117]
leaking cycles after trigger low: [135, 136, 151, 228, 234, 235, 237, 239, 240, 243, 244, 247, 249]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 8](figures/trigger-sweep/test8_halves.png)

## Test 9: trigger low after `qn = q * N`

In `barrett_reduce`, after `uint32_t qn = q * PARAM_N;`. The cut changes the instructions: the compiler normally computes `qn` and `r` with one `mls` (multiply and subtract), but here it uses `muls` before the call and `subs` after it. `r` leaves the window.

In [42]:
build_and_load(9)

In [43]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett-sweep/simpleserial-barrett-sweep-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b5f8      	push	{r3, r4, r5, r6, r7, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4c16      	ldr	r4, [pc, #88]	@ (4001b8 <handle_barrett+0x60>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	6023      	str	r3, [r4, #0]
  40016c:	4b13      	ldr	r3, [pc, #76]	@ (4001bc <handle_barrett+0x64>)
  40016e:	4605      	mov	r5, r0
  400170:	4798      	blx	r3
  400172:	6826      	ldr	r6, [r4, #0]
  400174:	4c12      	ldr	r4, [pc, #72]	@ (4001c0 <handle_barrett+0x68>)
  400176:	f244 5705 	movw	r7, #17669	@ 0x4505
  40017a:	fba4 3406 	umull	r3, r4, r4, r6
  40017e:	437c      	muls	r4, r7
  400180:	4b10      	ldr	r3, [pc, #64]	@ (4001c4 <handle_barrett+0x6c>)
  400182:	

Barrett part of the trigger window (`handle_barrett`, between the `blx` to `trigger_high` and the `blx` to `trigger_low`): `ldr` (read x), `ldr` (load mu), `movw` (load N), `umull`, `muls`, `ldr` (address of `trigger_low`).

In [44]:
traces, trigger_low = record()

trigger low at cycle 131


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [45]:
welch_halves(traces, trigger_low, "test9_halves")

leaking cycles before trigger low: [102, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121, 123]
leaking cycles after trigger low: [132, 134, 135, 148, 149, 150, 151, 152, 156, 157, 227, 228, 229, 237, 238, 240, 241, 243, 245, 248, 249]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 9](figures/trigger-sweep/test9_halves.png)

## Test 10: trigger low after `q = (x * mu) >> 32`

In `barrett_reduce`, after `uint32_t q = (uint32_t)(((uint64_t)x * PARAM_N_MU) >> 32);`. On the board, after `umull`. `qn` leaves the window.

In [46]:
build_and_load(10)

In [47]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett-sweep/simpleserial-barrett-sweep-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b570      	push	{r4, r5, r6, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4c16      	ldr	r4, [pc, #88]	@ (4001b8 <handle_barrett+0x60>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	6023      	str	r3, [r4, #0]
  40016c:	4b13      	ldr	r3, [pc, #76]	@ (4001bc <handle_barrett+0x64>)
  40016e:	4605      	mov	r5, r0
  400170:	4798      	blx	r3
  400172:	6826      	ldr	r6, [r4, #0]
  400174:	4c12      	ldr	r4, [pc, #72]	@ (4001c0 <handle_barrett+0x68>)
  400176:	fba4 3406 	umull	r3, r4, r4, r6
  40017a:	4b12      	ldr	r3, [pc, #72]	@ (4001c4 <handle_barrett+0x6c>)
  40017c:	4798      	blx	r3
  40017e:	f244 5205 	movw	r2, #17669	@ 0x4505
  400182:	fb02 6614 	ml

Barrett part of the trigger window (`handle_barrett`, between the `blx` to `trigger_high` and the `blx` to `trigger_low`): `ldr` (read x), `ldr` (load mu), `umull`, `ldr` (address of `trigger_low`).

In [48]:
traces, trigger_low = record()

trigger low at cycle 129


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [49]:
welch_halves(traces, trigger_low, "test10_halves")

leaking cycles before trigger low: [102, 104, 105, 113, 117]
leaking cycles after trigger low: [132, 133, 146, 147, 148, 149, 225, 226, 233, 234, 235, 236, 238, 239, 240, 241, 243, 245, 248, 249]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 10](figures/trigger-sweep/test10_halves.png)

## Test 11: trigger low after reading `x`

At the start of `barrett_reduce`. On the board, after the `ldr` that reads `x_in` and before `umull`. `q` leaves the window, so only the read of `x` is left.

In [50]:
build_and_load(11)

In [51]:
# Print the board instructions for the 'b' command
!arm-none-eabi-objdump --disassemble=handle_barrett {ELF}


simpleserial-barrett-sweep/simpleserial-barrett-sweep-CWHUSKY.elf:     file format elf32-littlearm


Disassembly of section .text:

00400158 <handle_barrett>:
  400158:	b538      	push	{r3, r4, r5, lr}
  40015a:	7883      	ldrb	r3, [r0, #2]
  40015c:	7842      	ldrb	r2, [r0, #1]
  40015e:	4c17      	ldr	r4, [pc, #92]	@ (4001bc <handle_barrett+0x64>)
  400160:	041b      	lsls	r3, r3, #16
  400162:	ea43 2302 	orr.w	r3, r3, r2, lsl #8
  400166:	7802      	ldrb	r2, [r0, #0]
  400168:	4313      	orrs	r3, r2
  40016a:	6023      	str	r3, [r4, #0]
  40016c:	4b14      	ldr	r3, [pc, #80]	@ (4001c0 <handle_barrett+0x68>)
  40016e:	4605      	mov	r5, r0
  400170:	4798      	blx	r3
  400172:	6824      	ldr	r4, [r4, #0]
  400174:	4b13      	ldr	r3, [pc, #76]	@ (4001c4 <handle_barrett+0x6c>)
  400176:	4798      	blx	r3
  400178:	4623      	mov	r3, r4
  40017a:	4a13      	ldr	r2, [pc, #76]	@ (4001c8 <handle_barrett+0x70>)
  40017c:	f244 5005 	movw	r0, #17669	@ 0x4505
  400180:	fba2 1203 	umull	r1, r2

Barrett part of the trigger window (`handle_barrett`, between the `blx` to `trigger_high` and the `blx` to `trigger_low`): `ldr` (read x), `ldr` (address of `trigger_low`).

In [52]:
traces, trigger_low = record()

trigger low at cycle 126


capture:   0%|          | 0/5000 [00:00<?, ?it/s]

In [53]:
welch_halves(traces, trigger_low, "test11_halves")

leaking cycles before trigger low: [6, 10, 19, 102, 109, 110, 111]
leaking cycles after trigger low: [143, 144, 222, 228, 231, 232, 233, 235, 236, 238, 239, 242, 243, 245, 246, 247, 248, 249]


:Overlay
   .Curve.I      :Curve   [cycles from trigger high]   (t)
   .Curve.II     :Curve   [cycles from trigger high]   (t)
   .Curve.Half_1 :Curve   [cycles from trigger high]   (t)
   .Curve.Half_2 :Curve   [cycles from trigger high]   (t)
   .VLine.I      :VLine   [x,y]

![Welch t-test on two halves, test 11](figures/trigger-sweep/test11_halves.png)

## Disconnect

Releases the Husky so other notebooks and programs can use it.

In [54]:
scope.dis()
target.dis()